## **HOUSE PRICES**

### **03 - Missing Data Indicator Feature**

Vẫn sử dụng `LotFrontage`, tạo thêm thuộc tính: `LotFrontage_missing`

Ví dụ:

| LotFrontage ban đầu | LotFrontage sau xử lý | LotFrontage_missing |
|:---:|:---:|:---:|
| 80 | 80 | 0 |
| NaN | Median | 1 |
| 70 | 70 | 0 |
| NaN | Median | 1 |

Trong đó:
- `0`: Giá trị ban đầu không bị thiếu.
- `1`: Giá trị ban đầu bị thiếu.

#### 1. Đọc dữ liệu

In [2]:
from pathlib import Path
import pandas as pd

DATA_PATH = Path("./data/train.csv")
df = pd.read_csv(DATA_PATH)

FEATURE = "LotFrontage"
SHOW_COLS = ["Id", FEATURE, "OverallQual", "SalePrice"]

print("So dong, so cot:", df.shape)
print("So gia tri thieu o LotFrontage:", df[FEATURE].isna().sum())

So dong, so cot: (1460, 81)
So gia tri thieu o LotFrontage: 259


#### 2. Quan sát dữ liệu ban đầu

In [3]:
missing_rows = df.index[df[FEATURE].isna()][:5]

print("5 dong ban dau co LotFrontage bi thieu:")
display(df.loc[missing_rows, ["Id", FEATURE, "OverallQual", "SalePrice"]])

5 dong ban dau co LotFrontage bi thieu:


,Id,LotFrontage,OverallQual,SalePrice
7,8,NaN,7,200000
12,13,NaN,5,144000
14,15,NaN,6,157000
16,17,NaN,6,149000
24,25,NaN,5,154000


#### 3. Tạo Missing Data Indicator Feature
Cần tạo Indicator trước khi điền khuyết để không mất thông tin về trạng thái thiếu ban đầu.

Quy ước
- `1` = giá trị bị thiếu trong dữ liệu gốc.
- `0` = giá trị không bị thiếu.

In [4]:
df_indicator = df.copy()
INDICATOR = FEATURE + "_missing"

# Tao cot danh dau truoc khi dien khuyet.
df_indicator[INDICATOR] = df_indicator[FEATURE].isna().astype(int)

print("Kiem tra so luong gia tri danh dau:")
print(df_indicator[INDICATOR].value_counts().sort_index())

Kiem tra so luong gia tri danh dau:
LotFrontage_missing
0    1201
1     259
Name: count, dtype: int64


#### 4. Điền giá trị thiếu bằng Median

In [5]:
median_value = df_indicator[FEATURE].median()
df_indicator[FEATURE] = df_indicator[FEATURE].fillna(median_value)

print(f"Median duoc dung: {median_value:.2f}")
print("25 dong dau tien cua train.csv sau khi xu ly:")
display(df_indicator[["Id", FEATURE, INDICATOR, "OverallQual", "SalePrice"]].head(25))

print("So gia tri thieu con lai o LotFrontage:", int(df_indicator[FEATURE].isna().sum()))
print("So dong indicator = 1:", int(df_indicator[INDICATOR].sum()))

Median duoc dung: 69.00
25 dong dau tien cua train.csv sau khi xu ly:


,Id,LotFrontage,LotFrontage_missing,OverallQual,SalePrice
0,1,65.0,0,7,208500
1,2,80.0,0,6,181500
2,3,68.0,0,7,223500
3,4,60.0,0,7,140000
4,5,84.0,0,8,250000
5,6,85.0,0,5,143000
6,7,75.0,0,8,307000
7,8,69.0,1,7,200000
8,9,51.0,0,7,129900
9,10,50.0,0,5,118000


So gia tri thieu con lai o LotFrontage: 0
So dong indicator = 1: 259


Ta có thấy, *LotFrontage* có 5 giá trị *Nan* ở những dòng dữ liệu id = [8, 13, 15, 17, 25] đã được thay thế bằng giá trị **Median = 69.00**, và đều có thêm cột **LotFrontage_missing = 1**.

In [6]:
output_dir = Path("output")
output_dir.mkdir(exist_ok=True)
output_path = output_dir / "missing_data_indicator.csv"
df_indicator.to_csv(output_path, index=False)
print("Da luu:", output_path.name)

Da luu: missing_data_indicator.csv
